# IMU Sensor Mapping Exploration

This notebook investigates which Movella DOT sensor corresponds to which body segment.

Known target body segments:
- left upper arm
- left forearm
- left hand
- right upper arm
- right forearm
- right hand
- sternum / trunk

Approach:
1. Do not use project docs/config as ground truth.
2. Scan multiple aligned H5 recordings.
3. Compare Movella gyro magnitude with ZED skeleton motion signals.
4. Infer candidate body segment mappings from the data only.
5. Report confidence conservatively.

In [1]:
from pathlib import Path
import itertools

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

DATASET_ROOT = Path("/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18")
ALIGNED_H5_ROOT = DATASET_ROOT / "aligned_h5"

SKIP_TASKS = {"Eye1", "Eye2", "Recalibration1"}

TARGET_BODY_SEGMENTS = [
    "left_hand",
    "left_forearm",
    "left_upper_arm",
    "right_hand",
    "right_forearm",
    "right_upper_arm",
    "trunk",
]

TARGET_BODY_SEGMENTS

['left_hand',
 'left_forearm',
 'left_upper_arm',
 'right_hand',
 'right_forearm',
 'right_upper_arm',
 'trunk']

## Helper Functions

In [2]:
def task_from_h5(path):
    stem = path.stem.replace("_aligned_dataset", "")
    participant_id = path.parent.name
    prefix = f"{participant_id}_"
    return stem[len(prefix):] if stem.startswith(prefix) else stem


def participant_from_h5(path):
    return path.parent.name


def is_calibration(path):
    return task_from_h5(path) in SKIP_TASKS


def list_adl_h5_files(limit_per_participant=4):
    selected = []
    for participant_dir in sorted(ALIGNED_H5_ROOT.iterdir()):
        if not participant_dir.is_dir():
            continue
        files = [p for p in sorted(participant_dir.glob("*.h5")) if not is_calibration(p)]
        selected.extend(files[:limit_per_participant])
    return selected


def finite_pair(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.isfinite(x) & np.isfinite(y)


def safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = finite_pair(x, y)
    if valid.sum() < 20:
        return np.nan
    x = x[valid]
    y = y[valid]
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def smooth(x, window=9):
    x = np.asarray(x, dtype=float)
    if window <= 1:
        return x
    s = pd.Series(x)
    return s.rolling(window=window, min_periods=1, center=True).mean().to_numpy()


def magnitude(df, cols):
    arr = np.column_stack([df[c] for c in cols])
    return np.sqrt(np.nansum(arr ** 2, axis=1))


def read_flat_group(h5_file, group_path):
    group = h5_file[group_path]
    return pd.DataFrame({key: group[key][:] for key in group.keys() if isinstance(group[key], h5py.Dataset)})


def select_main_skeleton_group(h5_file):
    candidates = [name for name in h5_file["modalities"].keys() if name.startswith("zed__skeleton__")]
    if not candidates:
        return None

    score_rows = []
    for name in candidates:
        group = h5_file["modalities"][name]
        score = 0
        for key in [
            "left_wrist_u", "left_wrist_v", "right_wrist_u", "right_wrist_v",
            "left_elbow_u", "left_elbow_v", "right_elbow_u", "right_elbow_v",
            "left_shoulder_u", "left_shoulder_v", "right_shoulder_u", "right_shoulder_v",
            "neck_u", "neck_v",
        ]:
            if key in group:
                values = group[key][:]
                score += int((np.isfinite(values) & (values >= 0)).sum())
        score_rows.append((score, name))

    return sorted(score_rows, reverse=True)[0][1]


def point_speed(df, prefix, xyz=True):
    suffixes = ["x", "y", "z"] if xyz else ["u", "v"]
    cols = [f"{prefix}_{s}" for s in suffixes]
    if not all(c in df.columns for c in cols):
        return None
    arr = np.column_stack([df[c].to_numpy(dtype=float) for c in cols])
    if not xyz:
        invalid = np.any(arr < 0, axis=1)
        arr[invalid] = np.nan
    diff = np.vstack([np.full((1, arr.shape[1]), np.nan), np.diff(arr, axis=0)])
    return np.sqrt(np.nansum(diff ** 2, axis=1))


def skeleton_motion_signals(skeleton_df):
    signals = {}
    for label, prefix in {
        "left_hand": "left_hand_index_1",
        "right_hand": "right_hand_index_1",
        "left_wrist": "left_wrist",
        "right_wrist": "right_wrist",
        "left_elbow": "left_elbow",
        "right_elbow": "right_elbow",
        "left_shoulder": "left_shoulder",
        "right_shoulder": "right_shoulder",
        "trunk": "spine_2",
        "neck": "neck",
    }.items():
        speed = point_speed(skeleton_df, prefix, xyz=True)
        if speed is not None:
            signals[label] = smooth(speed, window=9)
    return signals

## Files Used for Cross-Recording Validation

The notebook samples several non-calibration recordings per participant. You can increase `limit_per_participant` for a broader check.

In [3]:
h5_files = list_adl_h5_files(limit_per_participant=4)
len(h5_files), h5_files[:5]

(36,
 [PosixPath('/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18/aligned_h5/IA_4_H_02/IA_4_H_02_Brief_schreiben1_aligned_dataset.h5'),
  PosixPath('/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18/aligned_h5/IA_4_H_02/IA_4_H_02_Brief_schreiben2_aligned_dataset.h5'),
  PosixPath('/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18/aligned_h5/IA_4_H_02/IA_4_H_02_Chaos_several_Tasks_combine1_aligned_dataset.h5'),
  PosixPath('/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18/aligned_h5/IA_4_H_02/IA_4_H_02_Chaos_several_Tasks_combine2_aligned_dataset.h5'),
  PosixPath('/home/julius/datasets/seminar_object_detection/iAssistADL_dataset_export_2026-05-18/aligned_h5/IA_4_H_03/IA_4_H_03_Brief_schreiben1_aligned_dataset.h5')])

## Sensor Availability

In [4]:
availability_rows = []

for path in h5_files:
    with h5py.File(path, "r") as f:
        movella = sorted([name.replace("movella__", "") for name in f["modalities"].keys() if name.startswith("movella__DOT")])
        availability_rows.append({
            "participant_id": participant_from_h5(path),
            "task_name": task_from_h5(path),
            "movella_count": len(movella),
            "movella_ids": ", ".join(movella),
            "skeleton_group": select_main_skeleton_group(f),
        })

availability_df = pd.DataFrame(availability_rows)
availability_df

,participant_id,task_name,movella_count,movella_ids,skeleton_group
0,IA_4_H_02,Brief_schreiben1,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_139
1,IA_4_H_02,Brief_schreiben2,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_158
2,IA_4_H_02,Chaos_several_Tasks_combine1,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_42
3,IA_4_H_02,Chaos_several_Tasks_combine2,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_42
4,IA_4_H_03,Brief_schreiben1,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_2
5,IA_4_H_03,Brief_schreiben2,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_2
6,IA_4_H_03,Chaos_several_Tasks_combine1,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_2
7,IA_4_H_03,Chaos_several_Tasks_combine2,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_2
8,IA_4_H_04,Brief_schreiben1,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_76
9,IA_4_H_04,Brief_schreiben2,7,"DOT_D422CD008603, DOT_D422CD008CC7, DOT_D422CD...",zed__skeleton__Body_76


In [ ]:
all_ids = sorted(set(", ".join(availability_df["movella_ids"]).replace(",", "").split()))
all_ids

## Cross-Check With Skeleton Motion

This is an empirical sanity check, not a perfect calibration method. IMU angular velocity and skeleton point speed are different physical quantities, but strong side/segment relationships should still appear during many ADL movements.

In [ ]:
corr_rows = []

for path in h5_files:
    with h5py.File(path, "r") as f:
        sk_group = select_main_skeleton_group(f)
        if sk_group is None:
            continue

        skeleton_df = read_flat_group(f, f"modalities/{sk_group}")
        sk_signals = skeleton_motion_signals(skeleton_df)

        for modality_name in f["modalities"].keys():
            if not modality_name.startswith("movella__DOT"):
                continue

            dot_id = modality_name.replace("movella__", "")
            imu_df = read_flat_group(f, f"modalities/{modality_name}")
            gyro = smooth(magnitude(imu_df, ["gyro_x", "gyro_y", "gyro_z"]), window=9)

            for sk_label, sk_speed in sk_signals.items():
                corr_rows.append({
                    "participant_id": participant_from_h5(path),
                    "task_name": task_from_h5(path),
                    "dot_id": dot_id,
                    "skeleton_signal": sk_label,
                    "corr": safe_corr(gyro, sk_speed),
                })

corr_df = pd.DataFrame(corr_rows)
corr_df.head()

In [ ]:
corr_summary = (
    corr_df.groupby(["dot_id", "skeleton_signal"], as_index=False)
    .agg(mean_corr=("corr", "mean"), median_corr=("corr", "median"), n=("corr", "count"))
    .sort_values(["dot_id", "mean_corr"], ascending=[True, False])
)

corr_summary.head(30)

In [ ]:
top_corr = corr_summary.sort_values(["dot_id", "mean_corr"], ascending=[True, False]).groupby("dot_id").head(5)
top_corr

## Arm-Side Evidence

This groups skeleton signals into left/right/trunk evidence. It is useful for checking whether a sensor is more associated with the left or right side.

In [ ]:
SIDE_SIGNALS = {
    "left": {"left_hand", "left_wrist", "left_elbow", "left_shoulder"},
    "right": {"right_hand", "right_wrist", "right_elbow", "right_shoulder"},
    "trunk": {"trunk", "neck"},
}

side_rows = []
for side, signals in SIDE_SIGNALS.items():
    tmp = corr_df[corr_df["skeleton_signal"].isin(signals)]
    side_rows.append(
        tmp.groupby(["dot_id"], as_index=False)
        .agg(mean_corr=("corr", "mean"), median_corr=("corr", "median"), n=("corr", "count"))
        .assign(side_evidence=side)
    )

side_df = pd.concat(side_rows, ignore_index=True)
side_pivot = side_df.pivot_table(index=["dot_id"], columns="side_evidence", values="mean_corr")
side_pivot.reset_index()

In [ ]:
plot_df = side_pivot.reset_index().set_index("dot_id")[["left", "right", "trunk"]]
plot_df.plot(kind="bar", figsize=(13, 5))
plt.ylabel("Mean corr(IMU gyro magnitude, skeleton speed)")
plt.title("Side/trunk evidence by DOT sensor")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

## Data-Only Candidate Mapping and Confidence

This table is inferred from data only. It does not use project docs/config as ground truth.

Confidence is conservative:
- `medium`: a sensor clearly prefers left/right/trunk evidence, but exact segment may still be ambiguous.
- `low`: side/trunk evidence is weak or close between alternatives.

Natural ADL movements make upper arm, forearm, and hand move together, so exact distal/proximal labeling is harder than left/right/trunk separation.

In [ ]:
def side_to_candidate_segments(side):
    if side == "left":
        return "left_hand / left_forearm / left_upper_arm"
    if side == "right":
        return "right_hand / right_forearm / right_upper_arm"
    if side == "trunk":
        return "trunk / sternum"
    return "unknown"


def confidence_from_evidence(best_score, second_score, n_recordings):
    margin = best_score - second_score
    if n_recordings < 5 or not np.isfinite(best_score):
        return "low"
    if best_score >= 0.35 and margin >= 0.08:
        return "medium"
    if best_score >= 0.20 and margin >= 0.04:
        return "low-medium"
    return "low"


mapping_rows = []
side_table = side_pivot.reset_index()

for _, row in side_table.iterrows():
    dot_id = row["dot_id"]
    scores = {side: row.get(side, np.nan) for side in ["left", "right", "trunk"]}
    ranked = sorted(scores.items(), key=lambda kv: -np.inf if pd.isna(kv[1]) else kv[1], reverse=True)
    best_side, best_score = ranked[0]
    second_side, second_score = ranked[1]
    n_recordings = corr_df[corr_df["dot_id"] == dot_id][["participant_id", "task_name"]].drop_duplicates().shape[0]
    confidence = confidence_from_evidence(best_score, second_score, n_recordings)

    mapping_rows.append({
        "dot_id": dot_id,
        "data_inferred_side": best_side,
        "candidate_body_segments": side_to_candidate_segments(best_side),
        "confidence": confidence,
        "best_mean_corr": best_score,
        "second_best_side": second_side,
        "second_best_mean_corr": second_score,
        "margin": best_score - second_score,
        "n_recordings": n_recordings,
        "left_evidence_mean_corr": scores.get("left", np.nan),
        "right_evidence_mean_corr": scores.get("right", np.nan),
        "trunk_evidence_mean_corr": scores.get("trunk", np.nan),
    })

final_mapping_df = pd.DataFrame(mapping_rows).sort_values(["data_inferred_side", "best_mean_corr"], ascending=[True, False])
final_mapping_df

## Interpretation Notes

- This notebook intentionally ignores project docs/config mappings.
- Skeleton correlation can help check left vs right and trunk vs limb, but it cannot perfectly distinguish upper arm from forearm from hand because all linked arm segments often move together.
- The most uncertain distinction from ADL data alone is usually `upper_arm` vs `forearm` vs `hand` on the same side.
- The most reliable distinction from ADL data alone is usually `trunk` vs arms and left vs right side.
- For high-confidence exact segment labels, the best option is a calibration recording where only one body segment moves at a time.